# Practice 2 · Initialization in Code

| Marker | Meaning |
|---|---|
| 🟢 EDIT | you write this part |
| 🟡 READ | read it, no need to change |
| ⚪ RUN | just run it as it is |

### I. Background

A network initialised with all zeros cannot break symmetry; one initialised too large explodes,
one initialised too small vanishes. The lecture ended with two recipes:

- **Xavier**: keep the variance of every layer's output equal to its input. The lecture writes it
  with the fan_in only, *std = sqrt(1 / fan_in)*; *torch.nn.init.xavier_normal_* uses the symmetric
  version, *std = sqrt(2 / (fan_in + fan_out))* — that is why Q1 and Q2 give slightly different
  numbers.
- **Kaiming** (for ReLU): ReLU keeps only half of the values, so *std = sqrt(2 / fan_in)*

where *fan_in* is the number of inputs of a unit: *weight.shape[1]* for *Linear*,
*weight.shape[1] * kH * kW* for *Conv2d* (in short: *weight[0].numel()*).

### II. Task Requirements

1. **Q1** — write *apply_init(model, mode)* with *torch.nn.init*.
2. **Q2** — write *apply_init_manual(model, mode)* with *torch.randn* only.
3. Run the cells and compare the per-layer **activation std** of the three modes.

Steps: open the notebook in Haima, write the 🟢 parts, run the cells from top to bottom.

### III. Input and Output

| name | meaning | expected |
|---|---|---|
| *apply_init(model, "normal")* | weights ~ N(0, 0.01) | weight std about 0.01 |
| *apply_init(model, "xavier")* | *xavier_normal_* | weight std about 0.0245 in layer 1 |
| *apply_init(model, "kaiming")* | *kaiming_normal_* | weight std about 0.0255 in layer 1 |
| activation std, *normal* | last column of the print | the 2nd layer collapses, about 0.04 |
| activation std, *xavier / kaiming* | last column of the print | stays alive, about 0.6–0.8 |
| unknown mode | error | *ValueError* |

### IV. Who does what

- **Q1**: required for undergraduate students (postgraduates do it as well).
- **Q2**: required for postgraduate students; optional for undergraduates.
  An undergraduate who also finishes Q2 **gets bonus marks**; leaving it unfinished
  **costs nothing**.


## Question 1 · Three initialisations with *torch.nn.init*

Write *apply_init(model, mode)* that walks over every *Linear* / *Conv2d* module of the model and

- *mode="normal"*  → weights *~ N(0, 0.01)*
- *mode="xavier"*  → *torch.nn.init.xavier_normal_*
- *mode="kaiming"* → *torch.nn.init.kaiming_normal_* (ReLU)

and always sets every bias to zero. Any other mode must raise *ValueError*.
The function returns the model.

The given *layer_stds(model)* prints the std of every ReLU output for one fixed random input,
so everybody sees the same numbers. Watch the last column — one of the three modes collapses.


In [1]:
# 🟢 EDIT: fill in the body of apply_init() and run the cell.
# 🟡 READ: make_net() and layer_stds() are given above.
import math
import torch
import torch.nn as nn

torch.manual_seed(0)


# 🟡 READ: the network used by both questions
def make_net():
    """A plain MLP: 3072 -> 256 -> 128 -> 10."""
    return nn.Sequential(
        nn.Linear(3 * 32 * 32, 256), nn.ReLU(),
        nn.Linear(256, 128), nn.ReLU(),
        nn.Linear(128, 10),
    )


# 🟡 READ: std of every ReLU output for one fixed random input (same numbers for everybody)
def layer_stds(model, n=256, seed=123):
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(n, 3 * 32 * 32, generator=g)
    out = []
    with torch.no_grad():
        for m in model:
            x = m(x)
            if isinstance(m, nn.ReLU):
                out.append(round(float(x.std()), 4))
    return out


print(make_net())


def apply_init(model, mode):
    """Initialise every Linear/Conv2d weight, zero every bias, return the model."""
    # 🟢 CODE HERE - one loop over model.modules(), 4 parts:
    #   1) skip every module that is not nn.Linear or nn.Conv2d
    #   2) pick the initialiser by mode:
    #        "normal"  -> nn.init.normal_  with mean 0.0 and std 0.01
    #        "xavier"  -> nn.init.xavier_normal_
    #        "kaiming" -> nn.init.kaiming_normal_  (nonlinearity="relu")
    #        anything else -> raise ValueError
    #   3) if m.bias is not None: zero it with nn.init.zeros_
    #   4) return model
    if mode not in ("normal", "xavier", "kaiming"):
        raise ValueError(f"Unknown initialization mode: {mode}")
    for m in model.modules():
        if not isinstance(m, (nn.Linear, nn.Conv2d)):
            continue
        if mode == "normal":
            nn.init.normal_(m.weight, mean=0.0, std=0.01)
        elif mode == "xavier":
            nn.init.xavier_normal_(m.weight)
        elif mode == "kaiming":
            nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    return model


# ⚪ RUN: the three modes, weight std and activation std side by side
for mode in ("normal", "xavier", "kaiming"):
    net = apply_init(make_net(), mode)
    std = [round(float(l.weight.detach().std()), 4) for l in net if isinstance(l, nn.Linear)]
    print(f"{mode:>8}: weight std {std} | activation std {layer_stds(net)}")


Sequential(
  (0): Linear(in_features=3072, out_features=256, bias=True)
  (1): ReLU()
  (2): Linear(in_features=256, out_features=128, bias=True)
  (3): ReLU()
  (4): Linear(in_features=128, out_features=10, bias=True)
)
  normal: weight std [0.01, 0.01, 0.0099] | activation std [0.3263, 0.038]
  xavier: weight std [0.0245, 0.0723, 0.1241] | activation std [0.7946, 0.6779]
 kaiming: weight std [0.0255, 0.0885, 0.1253] | activation std [0.8195, 0.8047]


## Question 2 · The same thing, by hand

*torch.nn.init* hides the formula. Write *apply_init_manual(model, mode)* that applies the same
recipes by sampling with **torch.randn** (the point is to write the formula yourself, bias included):

- *mode="xavier"*  → *w = randn(shape) * sqrt(1 / fan_in)*
- *mode="kaiming"* → *w = randn(shape) * sqrt(2 / fan_in)*
- anything else    → *ValueError*

This xavier is the *fan_in only* form used in the lecture, so its std is a little smaller than
*torch.nn.init.xavier_normal_* (which uses fan_in + fan_out) — that is expected.
*fan_in* for a weight of shape *(out, in, ...)* is *weight[0].numel()* — it works for *Linear*
and for *Conv2d* alike. Do the assignment under *torch.no_grad()*.


In [2]:
# 🟢 EDIT: write apply_init_manual(), sampling every weight with torch.randn.
# 🟡 READ: make_net() and layer_stds() come from the Q1 cell above.
def apply_init_manual(model, mode):
    """Initialise the same way, but sample every weight with torch.randn."""
    # 🟢 CODE HERE - 5 parts:
    #   1) do everything inside  with torch.no_grad():   (the weights are leaf tensors)
    #   2) loop over model.modules(), keep nn.Linear / nn.Conv2d
    #   3) fan_in = m.weight[0].numel()
    #        std = math.sqrt(1 / fan_in)  for "xavier"
    #        std = math.sqrt(2 / fan_in)  for "kaiming"
    #        anything else -> raise ValueError
    #   4) m.weight.copy_(torch.randn(m.weight.shape) * std)   (copy_ keeps the same tensor)
    #   5) zero every bias:  m.bias.zero_()
    #   return model
    if mode not in ("xavier", "kaiming"):
        raise ValueError(f"Unknown initialization mode: {mode}")
    with torch.no_grad():
        for m in model.modules():
            if not isinstance(m, (nn.Linear, nn.Conv2d)):
                continue
            fan_in = m.weight[0].numel()
            if mode == "xavier":
                std = math.sqrt(1 / fan_in)
            else:
                std = math.sqrt(2 / fan_in)
            m.weight.copy_(torch.randn(m.weight.shape) * std)
            if m.bias is not None:
                m.bias.zero_()
    return model


# ⚪ RUN: the two modes, by hand
for mode in ("xavier", "kaiming"):
    net = apply_init_manual(make_net(), mode)
    std = [round(float(l.weight.detach().std()), 4) for l in net if isinstance(l, nn.Linear)]
    print(f"{mode:>8}: weight std {std} | activation std {layer_stds(net)}")


  xavier: weight std [0.018, 0.0628, 0.0853] | activation std [0.5857, 0.4243]
 kaiming: weight std [0.0255, 0.0884, 0.1226] | activation std [0.8238, 0.857]
